# 2.3 µm Iterative MFによる背景領域選択と人工プルーム二吸収帯評価

実メタンプルームを含むHISUIシーンに対して、次の順序で評価

1. **元シーンの2.3 µm帯にIterative Matched Filterを適用**し、実プルーム候補を背景統計から除外する
2. 2.3 µm帯のMF応答が背景中心に近く、空間的にも安定した場所を人工プルーム注入位置として自動選択する
3. 選んだ場所へ、MODTRAN絶対濃度LUTから作った相対放射輝度比を用いて人工プルームを注入する
4. 注入後のシーンに対して、1.6 µm帯と2.3 µm帯を**独立にIterative MF**で評価する
5. 2.3 µm帯の候補を1.6 µm帯が同一画素または近傍で支持するかを調べる
6. 既知の人工プルーム真値に対して、単一帯域と二吸収帯相互確認の性能を比較する

ここで「低い2.3 µm MF応答」は、最小の負値ではなく、**robust Z-scoreが0付近でメタンらしい正応答がない領域**と解釈。強い負の外れ値は、影・地表異常・校正誤差の可能性があるため注入場所には使わない。

主な評価対象は次の4つ。

- 1.6 µm帯MF単独
- 2.3 µm帯MF単独
- 二帯域の同一画素AND
- 2.3 µm候補を1.6 µm帯が近傍で支持する相互確認型MF

元シーンに実プルームがあるため、人工注入の評価は、元シーンで既に検出されていた候補を差し引いた**新規検出マスク**と、人工プルーム周辺の局所評価領域を用いて行う。

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, label, maximum_filter
from scipy.stats import rankdata
np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
# 入力
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"
OUTPUT_DIR = Path("./dual_window_background_injection_output")

# MODTRAN・HISUI
BACKGROUND_CH4_PPM = 1.8
FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5
UAS_STEP_PPM = 0.05

# Iterative MF 
MAX_ITERATIONS_BASELINE = 12
MAX_ITERATIONS_INJECTION = 10
EXCLUSION_Z_16 = 2.5
EXCLUSION_Z_23 = 2.5
EXCLUSION_DILATION_PIXELS = 2
MIN_BACKGROUND_FRACTION = 0.55
MIN_BACKGROUND_PIXELS = 300
CONVERGENCE_NEW_PIXEL_FRACTION = 2.5e-4
COVARIANCE_SHRINKAGE = 0.08
COVARIANCE_RIDGE_RELATIVE = 1e-8

# 人工プルーム注入場所の選択 
# Noneなら2.3 µm Iterative MFから自動選択。指定する場合はROI配列内の(row, col)
INJECTION_CENTER_YX = None
CENTER_SEARCH_STRIDE = 2
MAX_ABS_BASELINE_Z23 = 0.75
MAX_LOCAL_MAX_Z23 = 1.5
REAL_PLUME_EXCLUSION_BUFFER_PIXELS = 4

# 1.6 µm帯の既存異常が注入評価を汚さないための安全確認
# 主選択スコアは2.3 µm帯だけで作る
USE_16_AS_SAFETY_CHECK = False
MAX_ABS_BASELINE_Z16 = 1.5
MAX_LOCAL_ABS_Z16 = 2.0

# 人工プルーム形状 
PLUME_ANGLE_DEG = 0.0
PLUME_DECAY_PIX = 18.0
PLUME_CROSS_SIGMA_PIX = 4.0
PLUME_SOURCE_SIGMA_PIX = 2.0
PLUME_SUPPORT_FRACTION_FOR_LOCATION = 0.05

# LUT範囲外のピークは自動的に除外
INJECTION_PEAKS_PPM = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
PRIMARY_PEAK_PPM = 2.0

# 最終候補抽出 
DETECTION_Z_16 = 2.0
DETECTION_Z_23 = 3.0
NEIGHBORHOOD_RADIUS = 1
MIN_REGION_PIXELS = 3
BASELINE_CANDIDATE_BUFFER_PIXELS = 2

# 人工真値・局所評価
TRUE_MASK_FRACTION_OF_PEAK = 0.10
TRUE_MASK_MIN_ENHANCEMENT_PPM = 0.05
EVALUATION_BUFFER_PIXELS = 12
TOP_LOCATION_CANDIDATES = 30

# 主ピークでしきい値探索
THRESHOLDS_16 = np.arange(0.5, 4.01, 0.5)
THRESHOLDS_23 = np.arange(1.5, 5.01, 0.5)

RANDOM_SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. HISUI ROIスペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError("wave_***nm形式の列が見つかりません。")
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {value: i for i, value in enumerate(ys)}
    x_to_i = {value: i for i, value in enumerate(xs)}

    cube = np.full((len(ys), len(xs), spectra.shape[1]), fill_value, dtype=float)
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(cube, nodata_values=(0.0, -9999.0), require_positive=True):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)


df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_original, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_original)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_original.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)


## 3. MODTRAN絶対濃度LUTの読み込み、HISUI応答への畳み込み、UAS作成


In [ ]:
def load_ch4_absolute_concentration_lut(path):
    df_lut = pd.read_csv(path)
    candidates = [
        col for col in df_lut.columns
        if str(col).strip().lower() in {"wavelength", "wave", "wavelength_nm", "waveln"}
    ]
    if not candidates:
        raise ValueError("LUTに波長列が見つかりません。")

    wave_col = candidates[0]
    mod_wave = df_lut[wave_col].to_numpy(dtype=float)

    pairs = []
    for col in df_lut.columns:
        if col == wave_col:
            continue
        try:
            pairs.append((col, float(str(col).strip())))
        except ValueError:
            pass
    if len(pairs) < 2:
        raise ValueError("LUTに絶対CH4濃度の数値列が2列以上必要です。")

    pairs.sort(key=lambda item: item[1])
    concentration_grid = np.array([p[1] for p in pairs], dtype=float)
    lut_spectra = df_lut[[p[0] for p in pairs]].to_numpy(dtype=float).T
    order = np.argsort(mod_wave)
    return mod_wave[order], concentration_grid, lut_spectra[:, order]


def gaussian_srf_resample(mod_wave, mod_spectra, sensor_wave, fwhm_nm):
    mod_wave = np.asarray(mod_wave, dtype=float)
    mod_spectra = np.asarray(mod_spectra, dtype=float)
    sensor_wave = np.asarray(sensor_wave, dtype=float)

    if np.isscalar(fwhm_nm):
        fwhm = np.full(sensor_wave.shape, float(fwhm_nm))
    else:
        fwhm = np.asarray(fwhm_nm, dtype=float)
    if fwhm.shape != sensor_wave.shape:
        raise ValueError("FWHM配列の長さがsensor_waveと一致しません。")

    output = np.full((mod_spectra.shape[0], sensor_wave.size), np.nan)
    for j, center in enumerate(sensor_wave):
        sigma = fwhm[j] / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        use = np.abs(mod_wave - center) <= 4.0 * sigma
        if use.sum() < 2:
            output[:, j] = np.array([
                np.interp(center, mod_wave, spectrum)
                for spectrum in mod_spectra
            ])
        else:
            weights = np.exp(-0.5 * ((mod_wave[use] - center) / sigma) ** 2)
            weights /= weights.sum()
            output[:, j] = mod_spectra[:, use] @ weights
    return output


def interpolate_lut_spectrum(concentration, concentration_grid, sensor_lut):
    if not concentration_grid.min() <= concentration <= concentration_grid.max():
        raise ValueError(
            f"{concentration:.4f} ppmはLUT範囲外です。"
            f"LUT範囲: {concentration_grid.min():.4f}–{concentration_grid.max():.4f} ppm"
        )
    return np.array([
        np.interp(concentration, concentration_grid, sensor_lut[:, band_i])
        for band_i in range(sensor_lut.shape[1])
    ])


def compute_uas_from_absolute_lut(
    sensor_lut,
    concentration_grid,
    background_ppm,
    max_enhancement_ppm,
    step_ppm,
):
    max_allowed = concentration_grid.max() - background_ppm
    if max_enhancement_ppm > max_allowed:
        raise ValueError(
            f"UAS用増分がLUT範囲を超えます。使用可能最大値: {max_allowed:.4f} ppm"
        )

    enhancement_grid = np.arange(0.0, max_enhancement_ppm + 0.5 * step_ppm, step_ppm)
    enhancement_grid = np.unique(np.append(enhancement_grid, max_enhancement_ppm))

    background = interpolate_lut_spectrum(background_ppm, concentration_grid, sensor_lut)
    background = np.maximum(background, 1e-30)

    ratio_rows = []
    for enhancement in enhancement_grid:
        enhanced = interpolate_lut_spectrum(
            background_ppm + enhancement,
            concentration_grid,
            sensor_lut,
        )
        ratio_rows.append(enhanced / background)
    ratio_lut = np.asarray(ratio_rows)

    design = np.column_stack([np.ones_like(enhancement_grid), enhancement_grid])
    log_ratio = np.log(np.maximum(ratio_lut, 1e-30))
    coefficients, _, _, _ = np.linalg.lstsq(design, log_ratio, rcond=None)
    uas = -coefficients[1]
    return uas, enhancement_grid, ratio_lut


def build_enhancement_ratio_lut(
    sensor_lut,
    concentration_grid,
    background_ppm,
    maximum_enhancement_ppm,
    step_ppm=0.02,
):
    maximum_allowed = concentration_grid.max() - background_ppm
    if maximum_enhancement_ppm > maximum_allowed + 1e-12:
        raise ValueError(
            f"注入増分 {maximum_enhancement_ppm:.3f} ppm がLUT上限を超えます。"
            f"使用可能最大増分: {maximum_allowed:.3f} ppm"
        )

    grid = np.arange(0.0, maximum_enhancement_ppm + 0.5 * step_ppm, step_ppm)
    grid = np.unique(np.append(grid, maximum_enhancement_ppm))
    background = interpolate_lut_spectrum(background_ppm, concentration_grid, sensor_lut)
    background = np.maximum(background, 1e-30)

    rows = []
    for enhancement in grid:
        enhanced = interpolate_lut_spectrum(
            background_ppm + enhancement,
            concentration_grid,
            sensor_lut,
        )
        rows.append(enhanced / background)
    return grid, np.asarray(rows)


modtran_wavelengths, concentration_grid, modtran_spectra = (
    load_ch4_absolute_concentration_lut(CH4_LUT_CSV)
)
sensor_lut_absolute = gaussian_srf_resample(
    modtran_wavelengths,
    modtran_spectra,
    wavelengths,
    FWHM_NM,
)
uas_all, uas_enhancement_grid, uas_ratio_lut = compute_uas_from_absolute_lut(
    sensor_lut_absolute,
    concentration_grid,
    BACKGROUND_CH4_PPM,
    UAS_MAX_ENHANCEMENT_PPM,
    UAS_STEP_PPM,
)

mask_16 = (wavelengths >= WINDOW_16[0]) & (wavelengths <= WINDOW_16[1])
mask_23 = (wavelengths >= WINDOW_23[0]) & (wavelengths <= WINDOW_23[1])
if mask_16.sum() < 2 or mask_23.sum() < 2:
    raise ValueError(
        f"吸収窓内バンド数不足: 1.6 µm={mask_16.sum()}, 2.3 µm={mask_23.sum()}"
    )

maximum_allowed_enhancement = concentration_grid.max() - BACKGROUND_CH4_PPM
injection_peaks = np.array([
    value for value in INJECTION_PEAKS_PPM
    if 0 < value <= maximum_allowed_enhancement + 1e-12
], dtype=float)
if injection_peaks.size == 0:
    raise ValueError("LUT範囲内のINJECTION_PEAKS_PPMがありません。")
if PRIMARY_PEAK_PPM not in injection_peaks:
    PRIMARY_PEAK_PPM = float(injection_peaks[np.argmin(np.abs(injection_peaks - PRIMARY_PEAK_PPM))])
    warnings.warn(f"PRIMARY_PEAK_PPMをLUT範囲内の {PRIMARY_PEAK_PPM:.3f} ppmへ変更しました。")

enhancement_grid, enhancement_ratio_lut = build_enhancement_ratio_lut(
    sensor_lut_absolute,
    concentration_grid,
    BACKGROUND_CH4_PPM,
    float(injection_peaks.max()),
)

print("Absolute CH4 LUT grid [ppm]:", concentration_grid)
print("Injection peaks [ppm]:", injection_peaks)
print("Bands in 1.6 µm window:", int(mask_16.sum()))
print("Bands in 2.3 µm window:", int(mask_23.sum()))

plt.figure(figsize=(9, 4))
plt.plot(wavelengths[mask_16], uas_all[mask_16], label="1.6 µm UAS")
plt.plot(wavelengths[mask_23], uas_all[mask_23], label="2.3 µm UAS")
plt.xlabel("Wavelength [nm]")
plt.ylabel("UAS [1/ppm]")
plt.title("CH4 unit absorption spectrum after HISUI resampling")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
